# **Moving Load Analyser for Simply Supported Bridge**
## This tool evaluates the structural response of a simply supported bridge subjected to moving point loads.

### **Instructions:**
1.  **Point loads:** Provide the magnitudes of the loads, separated by commas.

2.  **Spacing:** Provide the spacing between successive loads, separated by commas.

3.  **Span length:** Specify the total span length of the bridge.

4.  **Point of interest:** Enter the location along the span where maximum forces are to be evaluated.

In [30]:
import pandas as pd
import numpy as np
from itertools import accumulate
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
from IPython.display import clear_output
from IPython.display import Image
from IPython.display import HTML
import matplotlib.animation as animation
import matplotlib as mpl
import ast
import ipywidgets as widgets
from IPython.display import display

In [31]:
## All Functions

# draw figure for beam span

def equilateral_triangle(apex, side_length, orientation="down"):

    h = np.sqrt(3)/2 * side_length
    x0, y0 = apex

    if orientation == "down":
        p1 = (x0 - side_length/2, y0 - h)
        p2 = (x0 + side_length/2, y0 - h)
    else:
        p1 = (x0 - side_length/2, y0 + h)
        p2 = (x0 + side_length/2, y0 + h)

    return np.array([p1, p2, apex, p1])

# function for box functions

def box_function(x, a, b):
    return np.where((x >= a) & (x <= b), 1, 0)

def box_function_1(x, a, b):
    return np.where((x > a) & (x <= b), 1, 0)

def box_function_2(x, a, b):
    return np.where((x >= a) & (x < b), 1, 0)

# define function for showing figure with beam span for any specific POI

def beam_with_point_loads(loads, spacings, span_length, position, poi_pos, description):

    if len(spacings) != len(loads) - 1:
        raise ValueError("Length of spacings must be one less than length of loads")

    # Beam endpoints
    p1 = (0, 0)
    p2 = (span_length, 0)

    fig, ax = plt.subplots(figsize=(10,5), dpi=150)

    # Beam line
    ax.plot([p1[0], p2[0]], [p1[1], p2[1]], 'k-', linewidth=2)

    # Supports
    tri1 = equilateral_triangle(p1, side_length=0.5, orientation="down")
    tri2 = equilateral_triangle(p2, side_length=0.5, orientation="down")
    ax.plot(tri1[:,0], tri1[:,1], 'b-')
    ax.plot(tri2[:,0], tri2[:,1], 'b-')

    # Arrow positions along beam (no scaling)
    arrows_x = [0]
    for s in spacings:
        arrows_x.append(arrows_x[-1] + s)

    sf_y = 0.2  # vertical scaling

    # Draw loads
    for i, h in enumerate(loads):
        x = arrows_x[i] + position   # shift horizontally by position
        ax.arrow(x, np.log2(h)*sf_y + 0.3, 0, -np.log2(h)*sf_y,
                 head_width=0.2, head_length=0.3,
                 fc='black', ec='black', linewidth=2)
        ax.text(x, np.log2(h)*sf_y + 0.05*np.log2(h)*sf_y + 0.3, f"{h} kN",
                ha='center', va='bottom', fontsize=10, color='black')

    if poi_pos is not None:
        circle = plt.Circle((poi_pos, 0), 0.3, color='red', fill=False, linewidth=2)
        ax.add_patch(circle)
        ax.text(poi_pos, -0.6, f"POI @ {poi_pos} m", ha='center', color='red', fontsize=9)


    # Formatting
    ax.set_xlim(-arrows_x[-1]-1, span_length+arrows_x[-1]+1)
    ax.set_ylim(-2, max(np.log2(loads))*sf_y + 2)
    ax.set_aspect('equal', adjustable='box')

    # Show x-axis and finer grid
    ax.get_yaxis().set_visible(False)   # hide y-axis only
    ax.set_xlabel("Beam length (m)")
    ax.grid(True, linestyle='--', alpha=0.6)

    # Set finer grid spacing
    ax.xaxis.set_major_locator(MultipleLocator(2))   # grid every 2 m
    ax.xaxis.set_minor_locator(MultipleLocator(0.2)) # finer grid every 0.5 m
    ax.grid(which='minor', linestyle=':', alpha=0.4) # lighter minor grid

    ax.set_title("POSITION FOR MAXIMUM"+description)
    plt.show()

# beam drawing function for animation
def beam_with_point_loads_1(loads, spacings, span_length, position, ax):
    if len(spacings) != len(loads) - 1:
        raise ValueError("Length of spacings must be one less than length of loads")

    ax.clear()
    p1, p2 = (0,0), (span_length,0)
    ax.plot([p1[0], p2[0]], [p1[1], p2[1]], 'k-', linewidth=2)

    # Supports
    tri1 = equilateral_triangle(p1, side_length=0.3, orientation="down")
    tri2 = equilateral_triangle(p2, side_length=0.3, orientation="down")
    ax.plot(tri1[:,0], tri1[:,1], 'b-')
    ax.plot(tri2[:,0], tri2[:,1], 'b-')

    # Arrow positions
    arrows_x = [0]
    for s in spacings:
        arrows_x.append(arrows_x[-1] + s)

    sf_y = 0.2
    for i, h in enumerate(loads):
        x = arrows_x[i] + position
        ax.arrow(x, np.log2(h)*sf_y + 0.3, 0, -np.log2(h)*sf_y,
                 head_width=0.2, head_length=0.3,
                 fc='black', ec='black', linewidth=2)
        ax.text(x, np.log2(h)*sf_y + 0.05*np.log2(h)*sf_y + 0.3, f"{h} kN",
                ha='center', va='bottom', fontsize=10, color='black')

    ax.set_xlim(-1, span_length+1)
    ax.set_ylim(-2, max(np.log2(loads))*sf_y + 2)
    ax.set_aspect('equal', adjustable='box')
    ax.get_yaxis().set_visible(False)
    ax.set_xlabel("Beam length (m)")
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.xaxis.set_major_locator(MultipleLocator(2))
    ax.xaxis.set_minor_locator(MultipleLocator(0.2))
    ax.grid(which='minor', linestyle=':', alpha=0.4)

def update(frame):
    pos_max_idx_bm = x_cord_1[frame]

    # Beam diagram
    beam_with_point_loads_1(load_list, spacing_list, span_1, pos_max_idx_bm, ax_beam)
    ax_beam.set_title(f"Moving Load Position = {pos_max_idx_bm:.2f} m")

    # Shear force diagram
    ax_sfd.clear()
    ax_sfd.plot(x_cord_span, sf_pos_poi[frame], 'b-', linewidth=2)
    ax_sfd.axhline(0, color='black', linewidth=1)
    ax_sfd.set_ylabel("Shear Force (kN)")
    ax_sfd.set_xlim(0, span_1)
    ax_sfd.grid(True, linestyle='--', alpha=0.6)

    # Bending moment diagram
    ax_bmd.clear()
    ax_bmd.plot(x_cord_span, bm_pos_poi[frame], 'r-', linewidth=2)
    ax_bmd.axhline(0, color='black', linewidth=1)
    ax_bmd.set_ylabel("Bending Moment (kNm)")
    ax_bmd.set_xlabel("Beam length (m)")
    ax_bmd.set_xlim(0, span_1)
    ax_bmd.grid(True, linestyle='--', alpha=0.6)

    return ax_beam, ax_sfd, ax_bmd

# function for sf and bm influence lines

def influence_all(x_vals, y_vals, span_1, load_list_last):

    sf_inf_all = []
    for y in y_vals:
# Vectorized computation for SF influence lines
        sf_x   = ( - x_vals / span_1) * box_function(x_vals, 0, y) + ( 1 - x_vals / span_1) * box_function(x_vals, y, span_1)
        sf_x   = np.concatenate([np.zeros(load_list_last), sf_x, np.zeros(load_list_last)])
        sf_inf_all.append(sf_x)
    bm_inf_all = []
    for y in y_vals:
# Computation for BM influence lines
        bm_x   = (1 - x_vals / span_1) * y - (y - x_vals) * box_function(x_vals, 0, y)
        bm_x   = np.concatenate([np.zeros(load_list_last), bm_x, np.zeros(load_list_last)])
        bm_inf_all.append(bm_x)

    return sf_inf_all, bm_inf_all

# Coloured print texts

def styled_print(text, color="green", bold=True):
    size   = 12
    weight = "bold" if bold else "normal"
    display(HTML(f"<span style='font-size:{size}pt; color:{color}; font-weight:{weight};'>{text}</span>"))


In [32]:
def beam_moving_load_analysis(span_1, spacing_list, load_list):
# calculate no of divisions on span with specified step size
    div_len  = 0.1 # in m
    no_div   = int((span_1)/div_len+1)
    load_len = sum(spacing_list)

# converting list to numpy array
    load_list           = np.array(load_list)
    spacing_list        = np.array(spacing_list)

# list of load position coordinates relative to left load
    positions           = np.concatenate(([0], np.cumsum(spacing_list)))

# convert positions array to finer step size
    load_position_list  = np.arange(0, positions[-1] + div_len, div_len)
    no_div_load         = len(load_position_list)

# create list of indices for finer positions array
    load_idx_finer      = np.arange(0, no_div_load, 1)

# create load list corresponding to finer positions array
    fine_loads          = np.zeros_like(load_position_list, dtype=float)
    indices             = np.searchsorted(load_position_list, positions)
    fine_loads[indices] = load_list

# create list of indices for finer positions array
    load_idx_finer      = np.arange(0, no_div_load, 1)

    indices_loads       = np.nonzero(fine_loads)[0]

# create and store influence line diagrams for different POI

    load_idx_list  = indices_loads
    load_list_last = load_idx_list[-1]
    tot_no_div     = (load_list_last * 2 + no_div)
    x_cord_span    = np.arange(0, span_1 + div_len, div_len)
    x_cord_span    = np.round(x_cord_span,3)

    x_cord         = np.arange(-load_list_last * div_len, (tot_no_div-load_list_last)* div_len, div_len)
    x_cord         = np.round(x_cord,3)

    step_no        = no_div + load_list_last
    x_cord_1       = x_cord[0:step_no]

# Create arrays of x and y positions
    x_vals = np.arange(no_div) * div_len
    y_vals = np.arange(no_div) * div_len
# Computation for SF influence lines

    sf_inf_all, bm_inf_all = influence_all(x_vals, y_vals, span_1, load_list_last)

# Convert lists to NumPy arrays
    bm_i_list         = []
    sf_i_list         = []
    pos_max_i_list_bm = []
    pos_max_i_list_sf = []
    bm_poi_pos        = []
    sf_poi_pos        = []
    for i in range(no_div): # looping across different POI
        poi_pos    = x_cord_span[i]

        bm_inf_i   = bm_inf_all[i]
        sf_inf_i   = sf_inf_all[i]

        bm_j_list = []
        sf_j_list = []
        for j in range(step_no): # looping across different load positions
            pos                         = x_cord[j]
            load_array                  = np.zeros(tot_no_div)
            load_array[j:j+no_div_load] = fine_loads

            bm_j                        = np.dot(load_array, bm_inf_i)
            bm_j                        = np.round(bm_j,3)
            bm_j_list.append(bm_j)

            sf_j                        = np.dot(load_array, sf_inf_i)
            sf_j                        = np.round(sf_j,3)
            sf_j_list.append(sf_j)

        bm_poi_pos.append(bm_j_list)
        pos_max_i_bm = x_cord[np.argmax(bm_j_list)]
        pos_max_i_list_bm.append(pos_max_i_bm)
        bm_i      = max(bm_j_list)
        bm_i_list.append(bm_i)

        sf_poi_pos.append(sf_j_list)
        pos_max_i_sf = x_cord[np.argmax(sf_j_list)]
        pos_max_i_list_sf.append(pos_max_i_sf)
        sf_i      = max(sf_j_list)
        sf_i_list.append(sf_i)

# draw beam and moving load position for maximum BM
    pos_max_all_bm     = pos_max_i_list_bm[np.argmax(bm_i_list)]
    poi_pos_max_bm     = x_cord_span[np.argmax(bm_i_list)]
    styled_print(f"Maximum BM = {max(bm_i_list)} kN-m <br>BM Max Occurs at x= {poi_pos_max_bm} m and When Last Load is Placed at x= {pos_max_all_bm} m")
    beam_with_point_loads(load_list, spacing_list, span_1, pos_max_all_bm, poi_pos_max_bm, " BENDING MOMENT ON OVERALL SPAN")

# draw beam and moving load position for maximum SF
    pos_max_all_sf     = pos_max_i_list_sf[np.argmax(sf_i_list)]
    poi_pos_max_sf     = x_cord_span[np.argmax(sf_i_list)]
    styled_print(f"Maximum SF = {max(sf_i_list)} kN <br>SF Max Occurs at x= {poi_pos_max_sf} m and When Last Load is Placed at x= {pos_max_all_sf} m")
    beam_with_point_loads(load_list, spacing_list, span_1, pos_max_all_sf, poi_pos_max_sf, " SHEAR FORCE ON OVERALL SPAN")

# draw beam and moving load position for maximum BM and SF about POI selected
    index           = np.where(x_cord_span == poi_required)[0][0]

    pos_max_idx_bm  = pos_max_i_list_bm[index]
    bm_max_idx      = bm_i_list[index]

    pos_max_idx_sf  = pos_max_i_list_sf[index]
    sf_max_idx      = sf_i_list[index]

    styled_print(f"With POI at x = {poi_required} m, Max BM is = {bm_max_idx} kN-m and Occurs When Last Load is Placed at x= {pos_max_idx_bm} m")
    beam_with_point_loads(load_list, spacing_list, span_1, pos_max_idx_bm, poi_required, " BENDING MOMENT @ POI")

    styled_print(f"With POI at x = {poi_required} m, Max SF is = {sf_max_idx} kN and Occurs When Last Load is Placed at x= {pos_max_idx_sf} m")
    beam_with_point_loads(load_list, spacing_list, span_1, pos_max_idx_sf, poi_required, " SHEAR FORCE @ POI")

# Take transpose of influence line matrices
    bm_pos_poi = list(map(list, zip(*bm_poi_pos)))
    sf_pos_poi = list(map(list, zip(*sf_poi_pos)))

    return ()


In [33]:
# Define widgets
loads    = widgets.Text(value="[120, 160, 80]", description="Loads:")
spacing  = widgets.Text(value="[2.5, 3.0]", description="Spacing:")
span     = widgets.FloatText(value=20.0, description="Span:")
poi_req  = widgets.FloatText(value=10.0, description="POI:")
run_btn  = widgets.Button(description="Run analysis")
out      = widgets.Output()

# Helper function
def parse_list(text):
    # safer than eval
    return ast.literal_eval(text)

# Callback function for button click
def run(_):
    with out:
        out.clear_output()
        # Save widget values into variables
        global load_list, spacing_list, span_1, poi_required
        load_list    = parse_list(loads.value)
        spacing_list = parse_list(spacing.value)
        span_1       = float(span.value)
        poi_required = float(poi_req.value)
        # Do operations on stored data
        beam_moving_load_analysis(span_1, spacing_list, load_list)

# Attach callback
run_btn.on_click(run)

# Display widgets + output area
display(loads, spacing, span, poi_req, run_btn, out)

Text(value='[120, 160, 80]', description='Loads:')

Text(value='[2.5, 3.0]', description='Spacing:')

FloatText(value=20.0, description='Span:')

FloatText(value=10.0, description='POI:')

Button(description='Run analysis', style=ButtonStyle())

Output()